# 02 · Vector Search endpoint and Delta Sync index over `code_excerpts`

1. Create-or-reuse our own STANDARD endpoint `bluerridge_vs` (never a shared one) and wait until it is ONLINE.
2. Create-or-reuse the Delta Sync index `mfg_mc_se_sa.bluerridge_agents.code_excerpts_vs`: primary key `citation_id`, managed embeddings from `databricks-bge-large-en` on column `text`, pipeline type TRIGGERED.
3. Trigger a sync, wait until the index is READY and the indexed row count equals the table's row count.
4. Smoke test: one natural-language query per project citation (top 3 each) plus an exact-label lookup, written to `outputs/sample_query.md`, with the pass bar asserted.

> **Disclaimer.** The five citation labels are fixed, illustrative project labels and clause numbers are not authoritative. ASME and IBC entries are plain-language paraphrases, not quotes of the copyrighted codes.

**Bundle or SDK?** `resources/vector-search.yml` declares the same endpoint and index (CLI v1.20.0 supports both resource types), so after `01_load_code_excerpts.ipynb` a `databricks bundle deploy -t demo` creates them. This notebook is written to be idempotent either way: it reuses whatever exists and only creates what is missing, so it also works in a workspace where the bundle was not deployed.

**Cost.** About 52 short texts are embedded once per sync; the work is trivial. Run it as a one-time serverless job.

In [1]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.16.0 which is incompatible.
googleapis-common-protos 1.62.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import time

from databricks.vector_search.client import VectorSearchClient

dbutils.widgets.text("catalog", "mfg_mc_se_sa")
dbutils.widgets.text("schema", "bluerridge_agents")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
ENDPOINT = "bluerridge_vs"
EMBED_ENDPOINT = "databricks-bge-large-en"
SRC_TABLE = f"{CATALOG}.{SCHEMA}.code_excerpts"
INDEX = f"{CATALOG}.{SCHEMA}.code_excerpts_vs"

vsc = VectorSearchClient(disable_notice=True)


def endpoint_state():
    try:
        return vsc.get_endpoint(ENDPOINT)["endpoint_status"]["state"]
    except Exception as e:
        if "NOT_FOUND" in str(e) or "does not exist" in str(e).lower() or "RESOURCE_DOES_NOT_EXIST" in str(e):
            return None
        raise


state = endpoint_state()
if state is None:
    print(f"endpoint {ENDPOINT} not found: creating STANDARD endpoint")
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")
else:
    print(f"endpoint {ENDPOINT} already exists (state {state}): reusing it")

deadline = time.time() + 40 * 60
while True:
    state = endpoint_state()
    print("endpoint state:", state)
    if state == "ONLINE":
        break
    assert time.time() < deadline, "endpoint did not reach ONLINE in 40 minutes"
    time.sleep(30)

endpoint bluerridge_vs already exists (state ONLINE): reusing it
endpoint state: ONLINE


In [3]:
ep = vsc.get_endpoint(ENDPOINT)
print({k: ep.get(k) for k in ("name", "endpoint_type", "num_indexes", "creator")}, "state =", ep["endpoint_status"]["state"])
assert ep["endpoint_type"] == "STANDARD" and ep["endpoint_status"]["state"] == "ONLINE"

{'name': 'bluerridge_vs', 'endpoint_type': 'STANDARD', 'num_indexes': 1, 'creator': 'joseph.hobbs@databricks.com'} state = ONLINE


In [4]:
existing = [i["name"] for i in vsc.list_indexes(ENDPOINT).get("vector_indexes", [])]
if INDEX in existing:
    print(f"index {INDEX} already exists: reusing it")
else:
    print(f"index {INDEX} not found: creating Delta Sync index (TRIGGERED, managed embeddings from {EMBED_ENDPOINT})")
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT,
        index_name=INDEX,
        source_table_name=SRC_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="citation_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBED_ENDPOINT,
    )
index = vsc.get_index(endpoint_name=ENDPOINT, index_name=INDEX)
print("index described:", {k: index.describe().get(k) for k in ("name", "index_type", "primary_key", "endpoint_name")})
spec = index.describe()["delta_sync_index_spec"]
print("delta_sync_index_spec:", {"source_table": spec["source_table"], "pipeline_type": spec["pipeline_type"], "embedding_source_columns": spec["embedding_source_columns"]})

index mfg_mc_se_sa.bluerridge_agents.code_excerpts_vs already exists: reusing it
index described: {'name': 'mfg_mc_se_sa.bluerridge_agents.code_excerpts_vs', 'index_type': 'DELTA_SYNC', 'primary_key': 'citation_id', 'endpoint_name': 'bluerridge_vs'}
delta_sync_index_spec: {'source_table': 'mfg_mc_se_sa.bluerridge_agents.code_excerpts', 'pipeline_type': 'TRIGGERED', 'embedding_source_columns': [{'name': 'text', 'embedding_model_endpoint_name': 'databricks-bge-large-en'}]}


## Sync and wait until the index holds every table row

In [5]:
table_rows = spark.table(SRC_TABLE).count()
print(f"source table rows = {table_rows}")


def wait_ready(timeout_s=40 * 60):
    deadline = time.time() + timeout_s
    while True:
        d = index.describe()["status"]
        print(f"ready={d.get('ready')}  indexed_row_count={d.get('indexed_row_count')}  detailed_state={d.get('detailed_state')}")
        if d.get("ready"):
            return d
        assert time.time() < deadline, "index did not become ready in time"
        time.sleep(20)


wait_ready()  # the first sync starts on its own when the index is created
index.sync()  # TRIGGERED pipeline: pick up any table change made since
time.sleep(15)
status = wait_ready()
deadline = time.time() + 20 * 60
while status.get("indexed_row_count") != table_rows and time.time() < deadline:
    time.sleep(20)
    status = wait_ready()
print("final status:", {k: status.get(k) for k in ("ready", "indexed_row_count", "detailed_state", "message")})
assert status["ready"] and status["indexed_row_count"] == table_rows, "indexed rows must equal table rows"

source table rows = 52
ready=True  indexed_row_count=52  detailed_state=ONLINE_NO_PENDING_UPDATE
ready=True  indexed_row_count=52  detailed_state=ONLINE_TRIGGERED_UPDATE
final status: {'ready': True, 'indexed_row_count': 52, 'detailed_state': 'ONLINE_TRIGGERED_UPDATE', 'message': 'Index is online but is currently is in the process of re-syncing initial data. Check latest status: https://fevm-mfg-mc-se-sa.cloud.databricks.com/explore/data/mfg_mc_se_sa/bluerridge_agents/code_excerpts_vs'}


In [6]:
ep = vsc.get_endpoint(ENDPOINT)
print(f"endpoint {ENDPOINT}: state={ep['endpoint_status']['state']}, type={ep['endpoint_type']}, indexes={ep['num_indexes']}")
print(f"index {INDEX}: ready={status['ready']}, indexed_row_count={status['indexed_row_count']} (table rows {table_rows}), embedding model {EMBED_ENDPOINT}, pipeline TRIGGERED")

endpoint bluerridge_vs: state=ONLINE, type=STANDARD, indexes=1
index mfg_mc_se_sa.bluerridge_agents.code_excerpts_vs: ready=True, indexed_row_count=52 (table rows 52), embedding model databricks-bge-large-en, pipeline TRIGGERED


## Smoke test

One plain-language query per project citation, top 3 each, then the exact-label lookup the app uses when a PM clicks a citation. **Pass bar** (set before looking at results, queries not tuned to fit): the matching primary citation is in the top 3 for at least 4 of the 5 queries, and the flange query must return `ASME B16.5 §4.2` in its top 3.

In [7]:
COLS = ["citation_id", "citation_label", "title", "source", "is_paraphrase"]
QUERIES = [
    ("flange pressure class too low for the line rating", "ASME B16.5 §4.2"),
    ("missing lockout tagout isolation tag on equipment", "OSHA 1910.147"),
    ("exit door too narrow for egress", "IBC 1005.1"),
    ("weld examination less than required", "ASME B31.3 §341"),
    ("no pressure relief valve shown", "ASME B31.3 §322"),
]
results = []
for q, expected in QUERIES:
    r = index.similarity_search(query_text=q, columns=COLS, num_results=3)
    hits = [(row[1], row[2], row[-1]) for row in r["result"]["data_array"]]  # label, title, score
    ok = expected in [h[0] for h in hits]
    results.append((q, expected, hits, ok))
    print(f"query: {q!r}  expected: {expected}  in top 3: {ok}")
    for rank, (label, title, score) in enumerate(hits, 1):
        print(f"  {rank}. {label:<22} {score:.4f}  {title}")

passed = sum(ok for *_, ok in results)
flange_ok = results[0][3]
print(f"\nprimary citation in top 3 for {passed}/5 queries; flange query returns ASME B16.5 §4.2 in top 3: {flange_ok}")
assert passed >= 4, "pass bar: at least 4 of 5 queries"
assert flange_ok, "pass bar: the flange query must return ASME B16.5 §4.2 in its top 3"

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
query: 'flange pressure class too low for the line rating'  expected: ASME B16.5 §4.2  in top 3: True
  1. ASME B16.5 §4.2        0.7290  Flange pressure class must suit design pressure and temperature
  2. ASME B16.5 §4.4        0.6854  Allowable pressure drops at higher temperature
  3. ASME B16.5 §4.1        0.6591  Flange pressure classes and what they mean
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
query: 'missing lockout tagout isolation tag on equipment'  expected: OSHA 1910.147  in top 3: True
  1. OSHA 1910.147          0.6757  Control of hazardous energy: lockout and tagout
  2. OSHA 1910.147(c)(5)    0.65

In [8]:
# Exact-label lookup: how the app resolves a clicked citation (a filter, not a semantic match)
LOOKUP = "OSHA 1910.147"
r = index.similarity_search(
    query_text="why did this drawing fail", columns=COLS + ["text"], num_results=5,
    filters={"citation_label": LOOKUP},
)
rows = r["result"]["data_array"]
print(f"filter citation_label = {LOOKUP!r} returned {len(rows)} row(s)")
for row in rows:
    print("  id:", row[0], "| label:", row[1], "| source:", row[3], "| paraphrase:", row[4])
    print("  text:", row[5])
assert len(rows) == 1 and rows[0][1] == LOOKUP

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
filter citation_label = 'OSHA 1910.147' returned 1 row(s)
  id: osha-1910-147 | label: OSHA 1910.147 | source: OSHA 29 CFR 1910 | paraphrase: True
  text: Control of hazardous energy: lockout and tagout. Before anyone services or maintains a machine or piece of equipment, the equipment must be isolated from all sources of hazardous energy and locked or tagged out so that it cannot start up or release stored energy. Each machine needs identified energy isolation points, such as breakers, valves and disconnects, where a lock or tag can be applied. A reviewer checks the drawing for lockout and tagout isolation points, tags or notes on each piece of equipment, and flags equipment that shows no lockout or isolation provision.


In [9]:
import os

DISCLAIMER = ("The five citation labels are fixed, illustrative project labels and clause numbers are not authoritative "
              "(in the real ASME B16.5, pressure-temperature ratings are in section 2, not 4.2). ASME and IBC entries "
              "are plain-language paraphrases, not quotes of the copyrighted codes.")
md = ["# Vector Search smoke test: `code_excerpts_vs`", "",
      f"> {DISCLAIMER}", "",
      f"Index `{INDEX}` on endpoint `{ENDPOINT}`; Delta Sync, TRIGGERED, managed embeddings from `{EMBED_ENDPOINT}` on column `text`; "
      f"{status['indexed_row_count']} rows indexed (table has {table_rows}). Top 3 results per query; score is the index's similarity score.", ""]
for q, expected, hits, ok in results:
    md += [f"## \"{q}\"", "", f"Expected primary citation: `{expected}` - **{'in top 3' if ok else 'NOT in top 3'}**", "",
           "| Rank | citation_label | title | score |", "|---|---|---|---|"]
    md += [f"| {i} | `{label}` | {title} | {score:.4f} |" for i, (label, title, score) in enumerate(hits, 1)]
    md += [""]
md += [f"**Pass bar:** matching primary citation in top 3 for {passed}/5 queries (need at least 4); flange query returns `ASME B16.5 §4.2` in top 3: {'yes' if flange_ok else 'no'}.", "",
       f"## Exact-label lookup (how the app resolves a clicked citation)", "",
       f"Filter `citation_label = '{LOOKUP}'` returned {len(rows)} row.", "",
       f"- `citation_id`: `{rows[0][0]}`", f"- `title`: {rows[0][2]}", f"- `source`: {rows[0][3]}, `is_paraphrase`: {rows[0][4]}", f"- text: {rows[0][5]}", ""]
os.makedirs("outputs", exist_ok=True)
open("outputs/sample_query.md", "w", encoding="utf-8").write("\n".join(md))
print("wrote outputs/sample_query.md")

wrote outputs/sample_query.md
